# Calling procedures by methods

Everything amoorion can do is a method: it has `Params`, a `run`, and front-ends
built from them. The same call looks the same from everywhere.

In [1]:
from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
Amoorion.init()


PosixPath('/tmp/amoorion-n32xdy11')

In [2]:
from amoorion.methods import METHODS

for name in METHODS.names():

    method = METHODS.get(name)

    print(
        f"{name:10}",
        'read-only' if method.read_only else 'changes  ',
        method.description
    )

accounts   read-only Which platforms have their secrets in the vault (values never shown)
automate   changes   Run rules on the messages of the inbox, once per message; a failure is tried again next time
compose    changes   Ask a language model (llm/url of the vault, any OpenAI-compatible server) to write something
delete     changes   Delete one message on its platform, keeping a copy in the dataset first
inbox      read-only Messages received so far, of every platform and hook, newest first
notify     changes   Send a notification to the platforms of AMOORION_NOTIFY; one result per platform
plugins    read-only List the platform plugins and what they can do
publish    changes   Publish one post on several platforms; one result per platform
read       read-only Read the latest messages of one platform (and keep them in the inbox and the dataset)
reply      changes   Answer one message on its platform
search     read-only Find messages of the inbox by their text, newest first
simulate

What a method takes is a model, so its schema is known to anything that asks:

In [3]:
METHODS.get(
    "publish"
).Params.model_json_schema()[
    "required"
]

['text', 'to']

From the object, by name or by `call`:

In [4]:
amo = Amoorion().use("dryrun")

amo.publish(
    text="by its name"
)[0]["ok"], 

amo.call(
    "publish", 
    text="by call"
)[0]["ok"]

True

Several as one procedure: `workflow` runs them in order and stops at the first
failure. `$last` is the text of the previous answer.

In [5]:
steps = [
    {
        "method": "publish", 
        "params": {
            "text": "the news", 
            "to": ["dryrun"]
        }
    },
    {
        "method": "publish", 
        "params": {
            "text": "repeated: $last", 
            "to": ["dryrun"]
        }
    },
    {
        "method": "publish", 
        "params": {
            "text": "this fails", 
            "to": ["nowhere"]
        }
    },
    {
        "method": "publish", 
        "params": {
            "text": "never runs", 
            "to": ["dryrun"]
        }
    },
]

for step in amo.workflow(steps=steps):
    print(
        step["method"], 
        step["ok"], 
        step["error"] or step["result"][0]["message"]["text"]
    )

publish True the news
publish True repeated: the news
publish False no plugin named 'nowhere'
